# 01 — Data Exploration: COCO Dataset for Retail Shelf Monitoring

**Covers:** Foundation for all subsequent chapters — data shapes, class distributions, and augmentation choices that feed the ProductionCV pipeline.

**Notes chapters:**
- `notes/02-advanced-deep-learning/ch01-residual-networks/` — the detection backbone we'll feed this data into
- `notes/02-advanced-deep-learning/ch03-object-detection/` — why annotation format matters for anchor generation

**Goal:** Understand what the COCO-style annotations look like, diagnose class imbalance early, choose augmentations that won't distort bounding boxes, and verify that the `src/data.py` pipeline produces correctly-shaped tensors.

**Prerequisites:**
- Completed `notes/02-advanced-deep-learning/ch01-residual-networks/`
- Python environment with `torch`, `torchvision`, `albumentations`, `matplotlib`, `pycocotools`
- `config.yaml` in `exercises/02-advanced-deep-learning/`

**`QUICK_MODE`:** Set `True` to use a 100-image synthetic sample (no download needed). Set `False` for real COCO data.

In [ ]:
# ── Imports & configuration ──────────────────────────────────────────────────
# sys.path ensures we can import from src/ regardless of where Jupyter is launched
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), '02-advanced-deep-learning'))

import json
import random
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import torch
import torchvision
from pathlib import Path
from collections import Counter

# QUICK_MODE=True uses a synthetic 100-image sample so this notebook runs
# without downloading the full COCO dataset (~20 GB).
# Set QUICK_MODE=False when you have real COCO annotations at DATA_DIR.
QUICK_MODE = False
DATA_DIR = Path('../data/coco')  # adjust if your COCO root is elsewhere
SAMPLE_SIZE = 100 if QUICK_MODE else None  # None = load all

random.seed(42)
np.random.seed(42)
print(f'PyTorch {torch.__version__} | torchvision {torchvision.__version__}')
print(f'QUICK_MODE={QUICK_MODE}, DATA_DIR={DATA_DIR}')

## 1. Dataset Overview — What Are We Working With?

COCO stores annotations as a single JSON file with five top-level keys:
- **`images`** — image metadata (id, filename, width, height)
- **`annotations`** — per-object bounding boxes (`[x, y, w, h]`), segmentation polygons, category_id
- **`categories`** — integer id → name mapping

For **retail shelf monitoring** we typically work with a *subset* of COCO categories (products, shelves, price tags) or a custom COCO-format annotation file. Here we load whichever annotation file is present (or generate a synthetic one in QUICK_MODE) and inspect its structure.

**Why check this first?** Corrupted JSON, missing image_ids, or duplicate annotation_ids will silently produce wrong training data if not caught here.

In [ ]:
# ── Load or generate COCO annotations ────────────────────────────────────────
ANNO_PATH = DATA_DIR / 'annotations' / 'instances_train2017.json'

if QUICK_MODE or not ANNO_PATH.exists():
    # --- Synthetic COCO-format data for QUICK_MODE ---
    # This mirrors the exact dict structure of a real COCO JSON so all
    # downstream code (pycocotools, our data.py) works without modification.
    CATEGORIES = [
        {'id': 1, 'name': 'product', 'supercategory': 'retail'},
        {'id': 2, 'name': 'shelf',   'supercategory': 'retail'},
        {'id': 3, 'name': 'price_tag','supercategory': 'retail'},
        {'id': 4, 'name': 'empty_slot','supercategory': 'retail'},
        {'id': 5, 'name': 'label',   'supercategory': 'retail'},
    ]
    images, annotations = [], []
    anno_id = 1
    for img_id in range(1, 101):
        w, h = random.choice([640, 800, 1024]), random.choice([480, 600, 768])
        images.append({'id': img_id, 'file_name': f'img_{img_id:05d}.jpg',
                        'width': w, 'height': h})
        for _ in range(random.randint(1, 12)):
            bx = random.randint(0, w - 40)
            by = random.randint(0, h - 40)
            bw = random.randint(20, min(200, w - bx))
            bh = random.randint(20, min(200, h - by))
            cat_id = random.choices([1, 2, 3, 4, 5], weights=[50, 15, 20, 10, 5])[0]
            annotations.append({
                'id': anno_id, 'image_id': img_id, 'category_id': cat_id,
                'bbox': [bx, by, bw, bh], 'area': bw * bh,
                'iscrowd': 0, 'segmentation': []
            })
            anno_id += 1
    coco_data = {'images': images, 'annotations': annotations, 'categories': CATEGORIES}
    print(f'[QUICK_MODE] Generated synthetic dataset: {len(images)} images, {len(annotations)} annotations')
else:
    with open(ANNO_PATH) as f:
        coco_data = json.load(f)
    images = coco_data['images']
    annotations = coco_data['annotations']
    CATEGORIES = coco_data['categories']
    if SAMPLE_SIZE:
        sampled_ids = {img['id'] for img in random.sample(images, SAMPLE_SIZE)}
        images = [img for img in images if img['id'] in sampled_ids]
        annotations = [a for a in annotations if a['image_id'] in sampled_ids]
    print(f'Loaded {len(images)} images, {len(annotations)} annotations')

cat_id_to_name = {c['id']: c['name'] for c in CATEGORIES}
print(f'Categories ({len(CATEGORIES)}): {[c["name"] for c in CATEGORIES]}')
print(f'Annotations per image (mean): {len(annotations)/len(images):.1f}')

## 2. Label Distribution — Class Balance and Rare Objects

**Why this matters:** If 80% of annotations are `product` and 1% are `empty_slot`, a naive detector will almost never predict `empty_slot` — which is precisely the class the retail client cares most about (out-of-stock detection). We need to know *before training* so we can:
1. Apply class-weighted loss (focal loss helps here — see notes ch03)
2. Oversample rare classes during augmentation
3. Set per-class confidence thresholds at inference time

In [ ]:
# ── Category distribution bar chart ──────────────────────────────────────────
# Count annotations per category
cat_counts = Counter(a['category_id'] for a in annotations)
names  = [cat_id_to_name[cid] for cid in sorted(cat_counts)]
counts = [cat_counts[cid]      for cid in sorted(cat_counts)]
total  = sum(counts)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Absolute counts
bars = axes[0].bar(names, counts, color='steelblue', edgecolor='white')
axes[0].set_title('Annotation Count per Category', fontsize=13)
axes[0].set_ylabel('Count')
for bar, cnt in zip(bars, counts):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + total*0.005,
                 str(cnt), ha='center', va='bottom', fontsize=9)

# Percentage breakdown — highlights imbalance
axes[1].pie(counts, labels=names, autopct='%1.1f%%',
            colors=plt.cm.Set2.colors[:len(names)])
axes[1].set_title('Class Share (%)', fontsize=13)

plt.tight_layout()
plt.suptitle('Label Distribution (imbalance drives loss weighting strategy)', y=1.02, fontsize=11)
plt.show()

# Imbalance ratio: max/min (>10x usually needs focal loss)
imbalance_ratio = max(counts) / min(counts)
print(f'Imbalance ratio (max/min): {imbalance_ratio:.1f}x')
if imbalance_ratio > 10:
    print('⚠  Severe imbalance detected — consider focal loss or oversampling')

## 3. Bounding Box Analysis — Size and Aspect Ratio Distributions

**Why this matters for anchor design:**
Faster R-CNN and YOLO use *anchor boxes* whose sizes are tuned to the expected object sizes in your dataset. If your objects are mostly small (products on a shelf, from a distance camera), you need **small anchors** and possibly **multi-scale FPN heads**. The aspect ratio histogram tells you how many anchor shapes you need.

Key observations to look for:
- Very small boxes (<32×32) → add a P2 FPN level
- Very wide boxes (aspect > 3) → wide anchor shapes needed
- Bimodal distribution → two distinct object types at different scales

In [ ]:
# ── Bounding box size & aspect ratio ─────────────────────────────────────────
bboxes = np.array([a['bbox'] for a in annotations], dtype=np.float32)  # [x, y, w, h]
widths  = bboxes[:, 2]
heights = bboxes[:, 3]
areas   = widths * heights
aspect_ratios = widths / (heights + 1e-6)  # avoid div-by-zero for degenerate boxes

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Scatter: width vs height (log-scale reveals scale clusters)
scatter = axes[0].scatter(widths, heights, alpha=0.3, s=8,
                          c=[a['category_id'] for a in annotations],
                          cmap='tab10')
axes[0].set_xlabel('BBox Width (px)'); axes[0].set_ylabel('BBox Height (px)')
axes[0].set_title('Width vs Height (log scale)')
axes[0].set_xscale('log'); axes[0].set_yscale('log')
plt.colorbar(scatter, ax=axes[0], label='category_id')

# Area histogram
axes[1].hist(np.sqrt(areas), bins=40, color='coral', edgecolor='white')
axes[1].set_xlabel('sqrt(area) — effective object scale (px)')
axes[1].set_ylabel('Count')
axes[1].set_title('Object Scale Distribution')
axes[1].axvline(32, color='navy', linestyle='--', label='32px (small)')
axes[1].axvline(96, color='green', linestyle='--', label='96px (medium)')
axes[1].axvline(256, color='red', linestyle='--', label='256px (large)')
axes[1].legend(fontsize=8)

# Aspect ratio histogram
axes[2].hist(np.log2(aspect_ratios + 1e-6), bins=40, color='mediumpurple', edgecolor='white')
axes[2].set_xlabel('log2(width/height)  [0=square, 1=2:1, -1=1:2]')
axes[2].set_ylabel('Count')
axes[2].set_title('Aspect Ratio Distribution (log2 scale)')
axes[2].axvline(0, color='red', linestyle='--', label='square')
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.show()

print(f'Area  — min: {areas.min():.0f}px², median: {np.median(areas):.0f}px², max: {areas.max():.0f}px²')
print(f'Small objects (<32²px): {(areas < 32**2).sum()} ({100*(areas < 32**2).mean():.1f}%)')

## 4. Augmentation Effects — What Transforms Are Safe?

**The augmentation contract for detection:**
- Geometric transforms (flip, crop, rotate) **must** transform bounding boxes and masks together — `albumentations` handles this automatically via its `bbox_params`.
- Color transforms (jitter, blur, CLAHE) are *image-only* and always safe.
- **Dangerous** for detection: random erasing that covers large objects, aggressive crops that shrink boxes below 10px.

The `src/features.py` `AugmentationPipeline` uses `albumentations`. Here we visualize the effect of four common transforms side-by-side to build intuition before tuning the augmentation config.

In [ ]:
# ── Augmentation visual comparison (3×4 grid) ─────────────────────────────────
# We import albumentations directly here to stay self-contained.
# src/features.py wraps these same transforms — see AugmentationPipeline.
try:
    import albumentations as A
    from albumentations.pytorch import ToTensorV2
    HAS_ALBUMENTATIONS = True
except ImportError:
    HAS_ALBUMENTATIONS = False
    print('albumentations not installed. Run: pip install albumentations')

if HAS_ALBUMENTATIONS:
    # Create a synthetic 256×256 RGB image with a visible rectangle for the bbox
    def make_dummy_image(seed=0):
        rng = np.random.RandomState(seed)
        img = (rng.rand(256, 256, 3) * 255).astype(np.uint8)
        # Draw a fake product region so we can see bbox transform effects
        img[60:160, 80:180] = rng.randint(100, 255, (100, 100, 3), dtype=np.uint8)
        return img

    transforms = [
        ('Original',       A.NoOp()),
        ('HorizontalFlip', A.HorizontalFlip(p=1.0)),
        ('RandomCrop',     A.RandomCrop(height=200, width=200, p=1.0)),
        ('ColorJitter',    A.ColorJitter(brightness=0.4, contrast=0.4,
                                         saturation=0.4, hue=0.1, p=1.0)),
    ]

    fig, axes = plt.subplots(3, 4, figsize=(14, 10))
    for row in range(3):
        base_img = make_dummy_image(seed=row)
        # Reference bbox in COCO [x, y, w, h] for the rectangle we drew
        bbox = [80, 60, 100, 100]

        for col, (name, transform) in enumerate(transforms):
            pipeline = A.Compose(
                [transform],
                bbox_params=A.BboxParams(format='coco', label_fields=['labels'],
                                          min_visibility=0.3)
            )
            result = pipeline(image=base_img, bboxes=[bbox], labels=['product'])
            aug_img = result['image']
            aug_bboxes = result['bboxes']

            axes[row, col].imshow(aug_img)
            for bx, by, bw, bh in aug_bboxes:
                rect = mpatches.Rectangle((bx, by), bw, bh,
                                          linewidth=2, edgecolor='lime', facecolor='none')
                axes[row, col].add_patch(rect)
            axes[row, col].set_title(name if row == 0 else '', fontsize=10)
            axes[row, col].axis('off')

    plt.suptitle('Augmentation Effects — bbox auto-transformed (green = transformed box)',
                 fontsize=12, y=1.01)
    plt.tight_layout()
    plt.show()

## 5. Mask Visualization — Instance vs Semantic Segmentation

**Instance segmentation** (Mask R-CNN, used in ProductionCV ch06) gives each *individual object* a unique mask — two adjacent products get different colors. This is what we need for out-of-stock detection: we must separate touching bottles.

**Semantic segmentation** merges all objects of the same class into one mask — faster, but can't count overlapping instances.

Here we render instance masks overlaid on the image for 4 sample images. In QUICK_MODE the masks are empty polygons (since we didn't generate them), so we fall back to drawing bounding box masks.

In [ ]:
# ── Instance mask overlay on 4 sample images ─────────────────────────────────
# We use a deterministic colormap: each instance gets a unique hue so
# overlapping objects remain visually separable.
from matplotlib.colors import hsv_to_rgb

def instance_colormap(n):
    """Return n visually distinct RGBA colors using golden ratio hue spacing."""
    colors = []
    for i in range(n):
        hue = (i * 0.618033988749895) % 1.0  # golden ratio ensures separation
        rgb = hsv_to_rgb([hue, 0.75, 0.95])
        colors.append((*rgb, 0.45))  # 45% alpha for overlay readability
    return colors

def render_instance_masks(img_h, img_w, annos):
    """Overlay instance masks (or bbox fills if no polygon) onto a blank canvas."""
    canvas = np.zeros((img_h, img_w, 4), dtype=np.float32)  # RGBA
    colors = instance_colormap(len(annos))
    for anno, color in zip(annos, colors):
        segs = anno.get('segmentation', [])
        if segs and isinstance(segs[0], list) and len(segs[0]) >= 6:
            # Real polygon — rasterize with cv2 if available
            try:
                import cv2
                mask = np.zeros((img_h, img_w), dtype=np.uint8)
                for seg in segs:
                    pts = np.array(seg).reshape(-1, 2).astype(np.int32)
                    cv2.fillPoly(mask, [pts], 1)
                canvas[mask == 1] = color
            except ImportError:
                pass  # fall through to bbox fill
        # Fallback: fill the bbox region with a translucent color
        x, y, w, h = [int(v) for v in anno['bbox']]
        x2, y2 = min(x + w, img_w), min(y + h, img_h)
        canvas[y:y2, x:x2] = color
    return canvas

# Build a lookup: image_id → list of annotations
from collections import defaultdict
img_to_annos = defaultdict(list)
for a in annotations:
    img_to_annos[a['image_id']].append(a)

# Pick 4 images that have ≥3 annotations (richer to visualize)
rich_images = [img for img in images if len(img_to_annos[img['id']]) >= 3][:4]
if len(rich_images) < 4:
    rich_images = images[:4]  # fallback

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, img_meta in zip(axes.flat, rich_images):
    img_h, img_w = img_meta['height'], img_meta['width']
    annos = img_to_annos[img_meta['id']]

    # Create a grey background (real images would load from disk here)
    bg = np.ones((img_h, img_w, 3), dtype=np.float32) * 0.85
    mask_overlay = render_instance_masks(img_h, img_w, annos)

    # Composite: bg * (1 - alpha) + color * alpha
    alpha = mask_overlay[:, :, 3:4]
    composite = bg * (1 - alpha) + mask_overlay[:, :, :3] * alpha
    composite = np.clip(composite, 0, 1)

    ax.imshow(composite)
    # Draw bboxes on top
    for anno in annos:
        bx, by, bw, bh = anno['bbox']
        rect = mpatches.Rectangle((bx, by), bw, bh,
                                   linewidth=1.5, edgecolor='white', facecolor='none')
        ax.add_patch(rect)
        ax.text(bx + 2, by - 4, cat_id_to_name.get(anno['category_id'], '?'),
                color='white', fontsize=7, weight='bold')
    ax.set_title(f'{img_meta["file_name"]}  ({len(annos)} instances)', fontsize=9)
    ax.axis('off')

plt.suptitle('Instance Mask Overlay — each color = one object instance', fontsize=12)
plt.tight_layout()
plt.show()